In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [2]:
#براساس وبسایت نیروی برق
# تعرفه‌های مصرف برق بر اساس پله‌ها
tariffs = {
    'tier_1': {
        'limits': [50, 100, 150, 200, 250, 300],  # سقف مصرف در هر پله (کیلووات ساعت)
        'rates': [2000, 3000, 4000, 5000, 6000, 7000]  # قیمت هر کیلووات ساعت (ریال)
    },
    'tier_2': {
        'limits': [400, 500],  # مصرف بین 1.5 تا 2 برابر الگو (کیلووات ساعت)
        'rates': [8500, 9000]  # قیمت هر کیلووات ساعت (ریال)
    },
    'tier_3': {
        'limits': [600, 700],  # مصرف بین 2 تا 3 برابر الگو (کیلووات ساعت)
        'rates': [9500, 10000]  # قیمت هر کیلووات ساعت (ریال)
    },
    'tier_4': {
        'limits': [800, 1000],  # مصرف بیشتر از 3 برابر الگو (کیلووات ساعت)
        'rates': [10700, 11000]  # قیمت هر کیلووات ساعت (ریال)
    }
}

In [3]:
def calculate_electricity_bill(consumption):
    bill = 0
    # پله اول (تا الگو)
    if consumption <= 300:
        for i, limit in enumerate(tariffs['tier_1']['limits']):
            if consumption > limit:
                bill += (limit - (tariffs['tier_1']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_1']['rates'][i]
            else:
                bill += (consumption - (tariffs['tier_1']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_1']['rates'][i]
                return bill

    # پله دوم (1.5 تا 2 برابر الگو)
    if consumption <= 500:
        excess = consumption - 300
        for i, limit in enumerate(tariffs['tier_2']['limits']):
            if excess > limit:
                bill += (limit - (tariffs['tier_2']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_2']['rates'][i]
            else:
                bill += (excess - (tariffs['tier_2']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_2']['rates'][i]
                return bill

    # پله سوم (2 تا 3 برابر الگو)
    if consumption <= 700:
        excess = consumption - 500
        for i, limit in enumerate(tariffs['tier_3']['limits']):
            if excess > limit:
                bill += (limit - (tariffs['tier_3']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_3']['rates'][i]
            else:
                bill += (excess - (tariffs['tier_3']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_3']['rates'][i]
                return bill

    # پله چهارم (بیش از 3 برابر الگو)
    excess = consumption - 700
    for i, limit in enumerate(tariffs['tier_4']['limits']):
        if excess > limit:
            bill += (limit - (tariffs['tier_4']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_4']['rates'][i]
        else:
            bill += (excess - (tariffs['tier_4']['limits'][i-1] if i > 0 else 0)) * tariffs['tier_4']['rates'][i]
            return bill

    return bill

In [4]:
data = {
    'monthly_consumption': [368, 314, 268, 202],
    'previous_balance': [885243, 243, 285, 149],
    'bill_amount': [3228484, 885265, 1361472, 830136],
    'amount_due': [2343000, 885000, 1362000, 830000]
}

df = pd.DataFrame(data)

df['next_month_amount_due'] = df['amount_due'].shift(-1)

df = df.dropna()

In [5]:
X = df[['monthly_consumption', 'previous_balance', 'bill_amount', 'amount_due']]
y = df['next_month_amount_due']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.joblib')

['scaler.joblib']

In [15]:
mlp = MLPRegressor(hidden_layer_sizes=(50, 30), activation='relu', solver='adam', max_iter=2000, random_state=42)

mlp.fit(X_train_scaled, y_train)

joblib.dump(mlp, 'mlp_model.joblib')

y_pred = mlp.predict(X_test_scaled)

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [7]:
MAE = mean_absolute_error(y_test, y_pred)
MSE = mean_squared_error(y_test, y_pred)
print(f"Mean Absolute Error: {MAE}")
print(f"Mean Squared Error: {MSE}")

current_bill = {
    'monthly_consumption': 300,
    'previous_balance': 100000,
    'bill_amount': 900000,
    'amount_due': 850000
}

Mean Absolute Error: 302563137.04742783
Mean Squared Error: 9.154445189998059e+16


In [9]:
current_df = pd.DataFrame([current_bill])

current_scaled = scaler.transform(current_df)

predicted_amount_due = mlp.predict(current_scaled)

predicted_consumption = current_bill['monthly_consumption']
predicted_bill = calculate_electricity_bill(predicted_consumption)
print(f"HAZINEYE GHABZE BARCH BAR ASASE PISH BINIYE MASRAF: {predicted_bill} Rial")

HAZINEYE GHABZE BARCH BAR ASASE PISH BINIYE MASRAF: 1350000 Rial


In [25]:
print(df.columns)

Index(['monthly_consumption', 'previous_balance', 'bill_amount', 'amount_due',
       'next_month_amount_due'],
      dtype='object')
